# Inter-Coder Reliability — Step 2

Sentence-level reliability for the full social-group label inventory. Two expert coders produced the ground truth collegially; one (for 226 sentences two) independent coder(s) annotated each sentence without the ground truth being revised afterwards.

Reliability is reported as Krippendorff's α (Krippendorff 2004) with MASI distance (Passonneau 2006) to handle multi-label agreement. Each sentence is one unit holding the ground truth and the independent annotation(s), so sentences with two independent coders contribute three values. Sentences without any independent coder are excluded. A separate table reports coder-vs-coder α on the sentences annotated by two independent coders.

In [31]:
import ast, re
from collections import Counter
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../src/step_2').resolve()))
from annotation_labels import CODEBOOK, load_sentences

In [32]:
OUTPUT_DIR = Path('./icr/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Codebook, label normalisation and parsing

Defined in `src/step_2/annotation_labels.py` (shared with `figures/annotation_confusion_matrix.py`): the codebook, the mapping of legacy and variant label spellings to canonical labels (unknown labels fall back to `'others'`), and the parsing of ground-truth and coder spans.

## MASI distance and Krippendorff's α

In [36]:
def masi_distance(A: frozenset, B: frozenset) -> float:
    if A == B:
        return 0.0
    inter = len(A & B)
    union = len(A | B)
    jaccard = inter / union if union > 0 else 0.0
    m = 0.67 if (A <= B or B <= A) else (0.33 if inter > 0 else 0.0)
    return 1.0 - jaccard * m


def alpha_masi(units) -> float:
    """Krippendorff's α with MASI distance.

    `units` holds one tuple of label sets per sentence (ground truth and independent coder(s)).
    Units with fewer than two values are not pairable and are skipped.
    """
    units = [u for u in units if len(u) >= 2]
    if not units:
        return np.nan
    n = sum(len(u) for u in units)
    # Observed disagreement: all ordered value pairs within a unit, weighted by 1 / (m_u - 1)
    D_o = sum(
        sum(masi_distance(u[i], u[j]) for i in range(len(u)) for j in range(len(u)) if i != j) / (len(u) - 1)
        for u in units
    ) / n
    # Expected disagreement over all ordered value pairs (i != j), computed on distinct
    # label sets weighted by frequency (self-pairs have distance 0, so this is exact).
    counts = Counter(v for u in units for v in u)
    values = list(counts.items())
    D_e = sum(
        ca * cb * masi_distance(a, b)
        for a, ca in values for b, cb in values
    ) / (n * (n - 1))
    return np.nan if D_e == 0 else round(1.0 - D_o / D_e, 4)

## Load

In [37]:
sentences = load_sentences()
print(f'{len(sentences):,} sentences | independent coders per sentence:',
      sentences['coders'].str.len().value_counts().sort_index().to_dict())

9,629 sentences | independent coders per sentence: {1: 9403, 2: 226}


## ICR by outlet (ground truth vs independent coders)

In [38]:
def fmt_alpha(a) -> str:
    return '--' if pd.isna(a) else f'{round(a, 2) + 0.0:.2f}'  # + 0.0 avoids '-0.00'


def outlet_row(name: str, country: str, sub: pd.DataFrame) -> dict:
    return {
        'Outlet':                name,
        'Country':               country,
        'Krippendorff α (MASI)': alpha_masi(list(sub['values'])),
        '# Sentences':           len(sub),
    }


COUNTRIES = sorted(sentences['country'].dropna().unique())

outlet_rows = []
for country in COUNTRIES:
    sub_c = sentences[sentences['country'] == country]
    for outlet in sorted(sub_c['outlet'].unique(), key=str.lower):
        outlet_rows.append(outlet_row(outlet, country, sub_c[sub_c['outlet'] == outlet]))
    outlet_rows.append(outlet_row(f'{country} (all outlets)', country, sub_c))
outlet_rows.append(outlet_row('Overall', 'All', sentences))

df_outlets = pd.DataFrame(outlet_rows)
df_outlets.to_csv(OUTPUT_DIR / 'icr_step2_by_outlet.csv', index=False)

df_outlets

,Outlet,Country,Krippendorff α (MASI),# Sentences
0,Figaro,France,0.9270,988
1,Liberation,France,0.7761,290
2,Mediapart,France,0.7182,390
3,Monde,France,0.9153,1014
4,MondeDiplo,France,0.6944,397
5,Parisien,France,0.9333,994
6,France (all outlets),France,0.8760,4073
7,Bild,Germany,0.8299,914
8,FAZ,Germany,0.8431,961
9,Spiegel,Germany,0.8336,920


## ICR by specific category and country

Aggregate rows per broad category are computed the same way, restricting all annotations of a sentence to the labels of the broad category. The LaTeX table is split in two parts.

For each specific category (codebook label), all annotations of every sentence (ground truth and independent coders) are restricted to that label, and α (MASI) is computed over all sentences of the country (so a sentence where no annotation uses the label counts as agreement on its absence). With a single label, MASI reduces to a binary presence/absence distance. `# Ground-truth mentions` is the number of distinct sentences whose ground-truth annotation contains the label. α is undefined (`--`) when the label does not occur in a scope.

In [39]:
def category_stats(sub: pd.DataFrame, labels: frozenset) -> tuple[float, int]:
    units = [tuple(v & labels for v in u) for u in sub['values']]
    # N = sentences whose ground truth contains the category
    n_gt = sum(1 for g in sub['gt_labels'] if g & labels)
    return alpha_masi(units), n_gt


SCOPES = [(c, sentences[sentences['country'] == c]) for c in COUNTRIES] + [('All', sentences)]

# One aggregate row per broad category (labels restricted to the whole subcategory),
# followed by one row per specific category.
cat_rows = []
for subcat, lbls in CODEBOOK.items():
    for lbl_set, name in [(frozenset(lbls), '(aggregate)')] + [(frozenset([l]), l) for l in lbls]:
        row = {'Subcategory': subcat, 'Category': name}
        for scope, sub in SCOPES:
            a, n = category_stats(sub, lbl_set)
            row[f'{scope} α (MASI)'] = a
            row[f'{scope} # Ground-truth mentions'] = n
        cat_rows.append(row)

df_cat = pd.DataFrame(cat_rows)
df_cat.to_csv(OUTPUT_DIR / 'icr_step2_by_category_country.csv', index=False)


# LaTeX: split in two tables at the broad-category boundary that best balances row counts.
# Layout follows dictionary/05_make_latex_table.py (wrapped labels, spacer rows).
LABEL_LINE_WRAP_THRESHOLD = 24  # chars; labels longer than this get split onto 2 lines
LABEL_LINE_GAP = '-3pt'         # tighten \shortstack's default inter-line gap
GROUP_GAP_HEIGHT = '5pt'        # height of the blank spacer row before each broad category
ARRAY_STRETCH = '0.9'

# manual line breaks for labels where the 2-line auto-split is still too wide
MANUAL_LABEL_BREAKS = {
    'Terrorists, rebels, revolutionaries and/or movements of armed resistance': [
        'Terrorists, rebels,',
        'revolutionaries and/or',
        'movements of armed resistance',
    ],
    # broad class (bold row); other broad class names stay on one line
    'Gender, sexuality, and sociocultural characteristics': [
        'Gender, sexuality, and',
        'sociocultural characteristics',
    ],
}


def wrap_label(text: str) -> str:
    """Split long label names onto balanced lines via \\shortstack."""
    text = text[:1].upper() + text[1:]
    if text in MANUAL_LABEL_BREAKS:
        joined = f'\\\\[{LABEL_LINE_GAP}]'.join(MANUAL_LABEL_BREAKS[text])
        return f'\\shortstack[l]{{{joined}}}'
    if len(text) <= LABEL_LINE_WRAP_THRESHOLD:
        return text
    spaces = [i for i, c in enumerate(text) if c == ' ']
    mid = len(text) / 2
    split_at = min(spaces, key=lambda i: abs(i - mid))
    line1, line2 = text[:split_at].strip(), text[split_at:].strip()
    return f'\\shortstack[l]{{{line1}\\\\[{LABEL_LINE_GAP}]{line2}}}'


def value_cells(r, bold=False):
    cells = []
    for scope, _ in SCOPES:
        cells += [fmt_alpha(r[f'{scope} α (MASI)']), f'{r[f"{scope} # Ground-truth mentions"]:,}']
    return [rf'\textbf{{{c}}}' for c in cells] if bold else cells


groups = []
for subcat, sub_df in df_cat.groupby('Subcategory', sort=False):
    body = [' & '.join([rf'\rule{{0pt}}{{{GROUP_GAP_HEIGHT}}}'] + [''] * 2 * len(SCOPES)) + r' \\']
    for _, r in sub_df.iterrows():
        if r['Category'] == '(aggregate)':
            name = wrap_label(subcat) if subcat in MANUAL_LABEL_BREAKS else subcat
            body.append(' & '.join([rf'\textbf{{{name}}}'] + value_cells(r, bold=True)) + r' \\')
        elif len(sub_df) > 2:  # skip the specific row when it duplicates the aggregate (e.g. 'Others')
            body.append(' & '.join([r'\quad ' + wrap_label(r['Category'])] + value_cells(r)) + r' \\')
    groups.append(body)

sizes = np.cumsum([len(g) for g in groups])
split = int(np.argmin(np.abs(sizes - sizes[-1] / 2))) + 1

cells = [r'\textbf{All categories}']
for scope, sub in SCOPES:
    cells += [rf'\textbf{{{fmt_alpha(alpha_masi(list(sub["values"])))}}}', rf'\textbf{{{len(sub):,}}}']
total_row = [r'\midrule', ' & '.join(cells) + r' \\']

parts = [groups[:split], groups[split:]]
lines = []
for i, part in enumerate(parts, start=1):
    lines += [
        r'\begin{table}[htbp]',
        r'\centering',
        rf'\renewcommand{{\arraystretch}}{{{ARRAY_STRETCH}}}',
        r'\scriptsize',
        r'\begin{threeparttable}',
        rf'\caption{{Inter-coder reliability for step 2 by specific group label and country. Part {i}/{len(parts)}.}}',
        rf'\label{{tab:icr_step2_by_category_country_part{i}}}',
        r'\begin{tabular}{l ' + ' '.join(['r r'] * len(SCOPES)) + '}',
        r'\toprule',
        ' & '.join([''] + [rf'\multicolumn{{2}}{{c}}{{{s}}}' for s, _ in SCOPES]) + r' \\',
        ''.join(rf'\cmidrule(lr){{{2 + 2 * j}-{3 + 2 * j}}}' for j in range(len(SCOPES))),
        ' & '.join(['Group'] + [r'$\alpha$ & $N$'] * len(SCOPES)) + r' \\',
        r'\midrule',
    ]
    for body in part:
        lines += body
    if i == len(parts):
        lines += total_row
    lines += [
        r'\bottomrule',
        r'\end{tabular}',
        r'\begin{tablenotes}[flushleft]',
        r'\footnotesize',
        r'\item \textit{Note:} Bold rows aggregate the specific group labels of each broad class. '
        r"$\alpha$: Krippendorff's $\alpha$ (MASI distance) between the ground truth and the independent coder(s). "
        r'$N$: number of sentences whose ground truth contains the group.',
        r'\end{tablenotes}',
        r'\end{threeparttable}',
        r'\end{table}',
        '',
    ]
(OUTPUT_DIR / 'icr_step2_by_category_country.tex').write_text('\n'.join(lines), encoding='utf-8')

df_cat

,Subcategory,Category,France α (MASI),France # Ground-truth mentions,Germany α (MASI),Germany # Ground-truth mentions,All α (MASI),All # Ground-truth mentions
0,Socio-economic position,(aggregate),0.8521,154,0.8644,226,0.8594,380
1,Socio-economic position,lower class,0.7649,45,0.8484,29,0.7945,74
2,Socio-economic position,middle class,0.9373,16,0.9165,11,0.9284,27
3,Socio-economic position,upper class,0.7331,21,0.7223,23,0.7278,44
4,Socio-economic position,"capital owners, investors and shareholders",0.9413,62,0.8920,119,0.9084,181
...,...,...,...,...,...,...,...,...
62,Real estate ownership,real-estate owners,0.9716,51,0.9231,47,0.9475,98
63,Real estate ownership,tenants,0.9742,19,0.9777,22,0.9761,41
64,Real estate ownership,homeless,0.9543,21,1.0000,14,0.9721,35
65,Others,(aggregate),0.8661,1120,0.8414,1339,0.8532,2459


## Coder vs coder (sentences with two independent coders)

α between the two independent coders only (ground truth not included), on the sentences annotated by two independent coders.

In [40]:
double = sentences[sentences['coders'].str.len() == 2]

cvc_rows = []
for scope, sub in [(c, double[double['country'] == c]) for c in COUNTRIES] + [('All', double)]:
    pairs = sub['coders'].map(lambda cs: ' + '.join(cs)).value_counts()
    cvc_rows.append({
        'Country':               scope,
        'Krippendorff α (MASI)': alpha_masi([tuple(cl) for cl in sub['coder_labels']]),
        '# Sentences':           len(sub),
        'Coder pairs':           '; '.join(f'{p} ({n})' for p, n in pairs.items()),
    })

df_cvc = pd.DataFrame(cvc_rows)
df_cvc.to_csv(OUTPUT_DIR / 'icr_step2_coder_vs_coder.csv', index=False)
df_cvc

,Country,Krippendorff α (MASI),# Sentences,Coder pairs
0,France,0.5224,14,isabelle + selma (8); mathieu + selma (4); isa...
1,Germany,0.9172,212,celine + selma (200); isabelle + selma (4); el...
2,All,0.8959,226,celine + selma (200); isabelle + selma (12); e...


## By annotator (ground truth vs each independent coder)

One row per independent coder: α between the ground truth and that coder, over the sentences the coder annotated. Coders are anonymised as A1, A2, ..., with the same IDs as in the step 1 notebook (A6 and A7 only coded step 2).

In [ ]:
# Fixed annotator IDs shared by the step 1 and step 2 notebooks: step 1 annotators first
# (A1-A5), then the annotators who only worked on step 2 (A6-A7).
ANNOTATORS = ['elisa', 'isabelle', 'mathieu', 'selma', 'theres', 'celine', 'stella']
assert {c for cs in sentences['coders'] for c in cs} <= set(ANNOTATORS)
CODER_IDS = {name: f'A{i}' for i, name in enumerate(ANNOTATORS, start=1)}

annotator_rows = []
for name, aid in CODER_IDS.items():
    units, countries = [], set()
    for gt, coders, coder_labels, country in zip(sentences['gt_labels'], sentences['coders'],
                                                 sentences['coder_labels'], sentences['country']):
        if name in coders:
            units.append((gt, coder_labels[coders.index(name)]))
            countries.add(country)
    annotator_rows.append({
        'Annotator':             aid,
        'Country':               ', '.join(sorted(countries)),
        'Krippendorff α (MASI)': alpha_masi(units),
        'N':                     len(units),
    })

df_annotators = pd.DataFrame(annotator_rows)
df_annotators.to_csv(OUTPUT_DIR / 'icr_step2_by_annotator.csv', index=False)
df_annotators